# Data Preparation — Qwen3-TTS Fine-Tuning

Converts raw recordings into `finetune/train_with_codes.jsonl`.  
Run `03_finetune.ipynb` after this.

```
finetune/raw_recordings/   ← drop WAVs here
finetune/data/             ← chunks land here (auto-created)
```

## 1. Setup

In [1]:
import os, json, warnings, subprocess, sys, random
import torch, whisper, soundfile as sf
import IPython.display as ipd
from pathlib import Path
from pydub import AudioSegment
from pydub.silence import split_on_silence

warnings.filterwarnings("ignore")

os.environ["PATH"] = "C:/ffmpeg/bin;" + os.environ["PATH"]

# ── config ────────────────────────────────────────────────────────────────
RAW_DIR        = Path("finetune/raw_recordings")  # drop your WAV files here
DATA_DIR       = Path("finetune/data")
REF_AUDIO      = Path("audio/ref_en.wav")
OUT_JSONL      = Path("finetune/train_raw.jsonl")
OUT_CODES      = Path("finetune/train_with_codes.jsonl")
TARGET_SR      = 24_000
MIN_DURATION_S = 2.0
MAX_DURATION_S = 15.0
# ──────────────────────────────────────────────────────────────────────────

DATA_DIR.mkdir(parents=True, exist_ok=True)

## 2. Check Raw Recordings

In [2]:
raw_files = sorted(RAW_DIR.glob("*.wav"))
for f in raw_files:
    dur = sf.info(str(f)).duration
    print(f.name, "-", round(dur / 60, 1), "min")
print("Total files:", len(raw_files))

sample_audio.wav - 15.1 min
Total files: 1


## 3. Split into Chunks

In [3]:
MIN_SILENCE_MS    = 600
SILENCE_THRESH_DB = -40
KEEP_SILENCE_MS   = 100

utterances = []
for rec in raw_files:
    seg = AudioSegment.from_wav(str(rec))
    seg = seg.set_frame_rate(TARGET_SR)
    seg = seg.set_channels(1)
    chunks = split_on_silence(seg, min_silence_len=MIN_SILENCE_MS,
                               silence_thresh=SILENCE_THRESH_DB,
                               keep_silence=KEEP_SILENCE_MS)
    for i, chunk in enumerate(chunks):
        dur = len(chunk) / 1000
        if dur < MIN_DURATION_S or dur > MAX_DURATION_S:
            continue
        out = DATA_DIR / f"{rec.stem[:30]}_{i:04d}.wav"
        chunk.export(str(out), format="wav")
        utterances.append((out, dur))

print("Chunks:", len(utterances))

Chunks: 114


## 4. Transcribe Chunks with Whisper large-v3

In [4]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

whisper_model = whisper.load_model("large-v3", device=device)

kept = []
for i, (wav_path, dur) in enumerate(utterances, 1):
    audio = whisper.load_audio(str(wav_path))
    result = whisper_model.transcribe(audio, language="en",
                                      beam_size=5, temperature=0.0,
                                      condition_on_previous_text=False)
    text = result["text"].strip()
    if len(text) < 5:
        wav_path.unlink()
        continue
    kept.append((wav_path, dur, text))
    if i % 200 == 0:
        print("Progress:", i, "/", len(utterances))

utterances = kept
print("Kept:", len(utterances))

Kept: 113


## 5. Review Samples (spot-check 10 random)

In [5]:
samples = random.sample(utterances, min(10, len(utterances)))
for path, dur, text in samples:
    data, sr = sf.read(str(path))
    print(path.name, "-", round(dur, 1), "s")
    print(" ", text)
    ipd.display(ipd.Audio(data, rate=sr))

sample_audio_0160.wav - 3.9 s
  We're ready to help design smart cities that serve citizens better, and we want to


sample_audio_0030.wav - 3.4 s
  In yesterday's parade, we saw the pride and the diversity of this nation.


sample_audio_0027.wav - 11.5 s
  It has been a great honor to be the first American President to join you for Republic Day.


sample_audio_0118.wav - 2.4 s
  So for all these reasons, India and the United States


sample_audio_0156.wav - 4.1 s
  As India pursues more trade and investment, we want to be first in line.


sample_audio_0069.wav - 4.9 s
  When Dr. King came to India, he said that being here in Gandhi's land


sample_audio_0093.wav - 5.0 s
  Having thrown off colonialism, we created constitutions that began with the three same


sample_audio_0187.wav - 9.0 s
  And being global partners means confronting the urgent global challenge of climate change.


sample_audio_0168.wav - 3.1 s
  The United States welcomes a greater role for India in the Asia Pacific.


sample_audio_0154.wav - 7.2 s
  We need our young people healthy for their futures, and we can do it. We have the technology to do it.


## 6. Write train_raw.jsonl

In [6]:
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for path, dur, text in utterances:
        f.write(json.dumps({"audio": str(path.resolve()), "text": text,
                            "ref_audio": str(REF_AUDIO.resolve())}, ensure_ascii=False) + "\n")

## 7. Extract Audio Codes

In [7]:
ft_scripts = Path("finetune/scripts")
if not ft_scripts.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/QwenLM/Qwen3-TTS", str(ft_scripts)], check=True)
    subprocess.run(["git", "-C", str(ft_scripts), "sparse-checkout", "set", "finetuning"], check=True)

result = subprocess.run(
    [sys.executable, str(ft_scripts / "finetuning" / "prepare_data.py"),
     "--device",               "cuda:0",
     "--tokenizer_model_path", "Qwen/Qwen3-TTS-Tokenizer-12Hz",
     "--input_jsonl",          str(OUT_JSONL.resolve()),
     "--output_jsonl",         str(OUT_CODES.resolve())],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print(result.stderr[-2000:])
else:
    print("Entries:", len(OUT_CODES.read_text().splitlines()))

Entries: 113
